# 04 — Comparando modelos sem escrever muito código

            [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/04_comparacao_modelos_hiperparametros.ipynb)

            **Duração estimada:** 75–90 minutos  
            **Pré-requisitos:** Notebook 02 e noções de validação cruzada.

            ## Objetivos

            - comparar quatro algoritmos nas mesmas partições
- construir um leaderboard com múltiplas métricas e tempo
- ajustar hiperparâmetros somente no treino
- comparar o modelo padrão e ajustado no teste intacto

            ## Fonte e licença

            CDC Diabetes Health Indicators (UCI 891).

            Fonte UCI sob CC BY 4.0; cite o conjunto e sua publicação.

            > **Uso responsável:** Este material tem finalidade exclusivamente educacional. Os resultados não devem ser usados para diagnóstico, prognóstico, tratamento, gestão assistencial ou decisão de saúde pública sem validação adequada, análise de contexto e supervisão de profissionais qualificados.

## Preparação do ambiente

> Como reutilizar uma interface pequena e transparente?

In [ ]:
# Preparação reproduzível do ambiente (a instalação ocorre só se faltar pacote).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "flavioluizseixas/aprendizado-de-maquina-para-saude"
REPO_DIR = Path("/content") / REPO.split("/")[-1]
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    command = ["git", "clone", f"https://github.com/{REPO}.git", str(REPO_DIR)]
    if REPO_DIR.exists():
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    subprocess.run(command, check=True)
    os.chdir(REPO_DIR)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    project = next((p for p in candidates if (p / "src").exists()), Path.cwd())
    os.chdir(project)

packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<4', 'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1', 'sklearn': 'scikit-learn>=1.4,<2', 'requests': 'requests>=2.31,<3', 'ucimlrepo': 'ucimlrepo>=0.0.7,<1'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from src.config import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)
print(f"Ambiente pronto em {Path.cwd()} | Colab={IN_COLAB} | semente={RANDOM_STATE}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.model_selection import train_test_split

from src.data_loading import load_cdc_diabetes
from src.evaluation import classification_report_health
from src.model_selection import (
    compare_classifiers, get_default_classifiers,
    tune_classifier, tuning_results_frame,
)

FAST_MODE = True
data, _ = load_cdc_diabetes(20_000 if FAST_MODE else 40_000, random_state=RANDOM_STATE)

In [ ]:
X = data.drop(columns="Diabetes_binary")
y = data["Diabetes_binary"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("O teste foi reservado antes de qualquer comparação:", X_test.shape)

## Pergunta orientadora

> Qual modelo apresenta melhor equilíbrio entre ordenação, erros e custo computacional na validação?

## Experimento: comparação

> Todos os modelos viram exatamente os mesmos cinco folds?

In [ ]:
models = get_default_classifiers(random_state=RANDOM_STATE)
leaderboard = compare_classifiers(
    X_train, y_train, models=models, cv=5,
    scoring=["roc_auc", "f1", "recall", "precision"],
)
display(leaderboard.round(3))
assert (leaderboard["status"] == "ok").all(), leaderboard[["model", "status"]]

In [ ]:
metric_columns = ["roc_auc", "f1", "recall", "precision", "balanced_accuracy"]
leaderboard.set_index("model")[metric_columns].plot.bar(figsize=(11, 4), ylim=(0, 1), title="Leaderboard — média em cinco folds")
plt.ylabel("Métrica"); plt.xticks(rotation=20, ha="right"); plt.show()
leaderboard.set_index("model")["fit_time"].sort_values().plot.barh(title="Tempo médio de ajuste por fold")
plt.xlabel("Segundos"); plt.show()

### Como interpretar

O primeiro lugar depende da métrica. ROC-AUC mede ordenação; recall e precisão mostram trocas diferentes. Tempo também é parte do custo. Diferenças pequenas frente ao desvio-padrão pedem cautela.

## Busca de hiperparâmetros

> É possível ajustar sem consultar o teste?

In [ ]:
best_name = leaderboard.iloc[0]["model"]
best_model, search = tune_classifier(
    model_name=best_name, X=X_train, y=y_train,
    cv=5, n_iter=5 if FAST_MODE else 20,
    scoring="roc_auc", random_state=RANDOM_STATE,
)
print("Modelo:", best_name)
print("Melhores parâmetros:", search.best_params_)
display(tuning_results_frame(search).round(3))

## Avaliação final

> O ajuste melhorou em dados realmente reservados?

In [ ]:
default_model = get_default_classifiers(RANDOM_STATE)[best_name]
default_model.fit(X_train, y_train)
default_prob = default_model.predict_proba(X_test)[:, 1]
tuned_prob = best_model.predict_proba(X_test)[:, 1]
comparison = pd.DataFrame({
    "padrão": classification_report_health(y_test, default_prob >= 0.5, default_prob),
    "ajustado": classification_report_health(y_test, tuned_prob >= 0.5, tuned_prob),
})
display(comparison.loc[["roc_auc", "pr_auc", "f1", "sensibilidade", "especificidade"]].round(3))

In [ ]:
tuned_pred = (tuned_prob >= 0.5).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, tuned_pred, cmap="Blues", ax=axes[0])
axes[0].set_title("Matriz de confusão — ajustado")
RocCurveDisplay.from_predictions(y_test, default_prob, name="Padrão", ax=axes[1])
RocCurveDisplay.from_predictions(y_test, tuned_prob, name="Ajustado", ax=axes[1])
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set_title("ROC no teste final"); plt.tight_layout(); plt.show()

### Como interpretar

O teste foi usado uma única vez, depois da escolha e do ajuste. Uma melhora de validação pode não se repetir no teste. O resultado compara algoritmos neste experimento, não estabelece utilidade clínica.

## Limitações e responsabilidade

- A busca cobre espaços pequenos para caber no Colab; não é exaustiva.
- Múltiplas comparações aumentam o risco de escolher variações fortuitas.
- Desempenho agregado pode esconder diferenças por subgrupo e calibração inadequada.

## Atividade

Mude a métrica de ordenação do leaderboard para `recall`. Registre se o vencedor muda e compare precisão, tempo e variabilidade — sem tocar no teste durante a escolha.

## Três aprendizados principais

1. Comparações justas reutilizam os mesmos folds.
2. Acurácia sozinha é frágil em classes desbalanceadas.
3. Hiperparâmetros pertencem à validação; teste pertence somente ao final.

## Referências

- [UCI — CDC Diabetes Health Indicators](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)
- [scikit-learn — model selection](https://scikit-learn.org/stable/model_selection.html)
- [scikit-learn — RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)

## Versões das bibliotecas

Registre o ambiente junto ao resultado.

In [ ]:
from src.config import library_versions
library_versions(('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'ucimlrepo'))